In [92]:
# pandas import 
import pandas as pd
import numpy as np
import time
import re
from scipy import stats

path = "../data/raw/planlama.csv"

# Veriyi oku
df = pd.read_csv(
    path,
    sep=";",
    encoding="latin-1",
    decimal=",",
    parse_dates=["Tarih"],
    date_format="%d.%m.%Y",
    dtype={"id": int, "OrtalamaCevre_AG": float, "AxialTelYuksekligi_AG": float }
    )

def expand_trafo_no(val):
    # Senaryo 4: null veya 0
    if pd.isna(val) or str(val).strip() in ["0", ""]:
        return [0]

    val = str(val).strip()

    # Tüm özel ayraçları normalize et
    # ..., ---, / vs → -
    val = re.sub(r"\.{2,}|-{2,}|/", "-", val)

    # Eğer aralık varsa
    if "-" in val:
        try:
            start, end = val.split("-")
            start, end = int(start), int(end)

            # Senaryo 1-2-5
            return list(range(start, end + 1))
        except:
            return [val]  # beklenmeyen format fallback

    # Senaryo 3: tek değer
    try:
        return [int(val)]
    except:
        return [val]
    
df["TrafoNo"] = df["TrafoNo"].apply(expand_trafo_no)
# print(df.head())

# Çoklu sütun → DataFrame döner
df_expanded = df[["TrafoNo", "Tarih", "OrtalamaCevre_AG", "AxialTelYuksekligi_AG"]]
# print(df_expanded.head())

# iki tarih aralığında tüm dataları getir
start_date = "2023-01-01"
end_date = "2023-12-31"
mask = (df["Tarih"] >= start_date) & (df["Tarih"] <= end_date)
## filtered_df = df_expanded[mask]
# print(df[mask].head())

# TrafoNo dizilerinin eleman sayısının toplamını hesapla
df_trafo_dizi_mask = df[["ProjeSiparisNo", "TrafoNo"]]
df_trafo_dizi_mask["TrafoCount"] = df["TrafoNo"].apply(len)

trafo_coun_sum_max = df_trafo_dizi_mask["TrafoCount"].max()
# trafo_coun_sum_max değerinin sipariş numaralarıyla birlikte gösterilmesi
max_siparis_no = df_trafo_dizi_mask[df_trafo_dizi_mask["TrafoCount"] == trafo_coun_sum_max][["ProjeSiparisNo", "TrafoCount"]]
# print("max_siparis_no:" + max_siparis_no.to_string(index=False))
# print(df_trafo_dizi_mask.head())


# Özel aggregation fonksiyonu
def gelir_artisi(x):
    return x.max() - x.min()

# Güvenli ovallik_orani sütunu oluştur (sıfıra bölünmeyi önle)
dataYaz = np.where((df["DarCap"].isna()) | (df["DarCap"] == 0), df["GenisCap"], df["DarCap"])
df["ovallik_orani"] = df["GenisCap"] / dataYaz
# print(df[["GenisCap", "DarCap", "ovallik_orani"]].head())

ozet = df.groupby("Guc").agg(
    siparis_sayisi=("id", "count"),
    cekirdek_agirligi=("CekirdekAgirligi", "sum"),
    ovallik_orani=("ovallik_orani", "mean")
).query("Guc > 1000").fillna(0).query("Guc < 1200")
# ozet = ozet[ozet.index > 1000]
# filter ekle Guc değeri 1000'den büyük olanları göster
# print(ozet)


def eksik_raporu(df):
    eksik = df.isnull().sum()
    eksik_oran = df.isnull().mean() * 100
    rapor = pd.DataFrame({
        "Eksik Sayı": eksik,
        "Oran (%)":   eksik_oran.round(2),
        "Veri Tipi":  df.dtypes
    })
    return rapor[rapor["Eksik Sayı"] > 0].sort_values("Oran (%)", ascending=False)

# print(eksik_raporu(df))

df["darCapAgirlikMedyan"] = df.groupby("Guc")["DarCap"].transform(
    lambda x: x.fillna(x.median())
)
# print(df[["Guc", "DarCap", "darCapAgirlikMedyan"]].head())
    
# Kat_Sayisi değerileri Null olanları 1 olarak değiştir
df["Kat_Sayisi"] = df["Kat_Sayisi"].fillna(1)

df["DarCap"] = df["DarCap"].fillna(df["GenisCap"])

# print(eksik_raporu(df))

# Duplikasyonları tespit et
print(f"Toplam satır: {len(df['Guc'])}")
# tekrar eden Guc değerlerinin sayısı
gunTekrar = df.duplicated(subset=["Guc"]).sum()
# tekrar etmene guc değerleri kaç adet var
# print(f"Tekrarlayan Guc sayısı: {gunTekrar}")

# ProjeSiparisNo bazında tekrar edenleri yazdır
tekrarEdenSiparisNo = df[df.duplicated(subset=["ProjeSiparisNo"], keep=False)]["ProjeSiparisNo"].unique()
# print(f"Tekrarlayan ProjeSiparisNo sayısı: {(tekrarEdenSiparisNo)}")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

tekrar_edenler = df[df["ProjeSiparisNo"].duplicated(keep=False)].sort_values("ProjeSiparisNo")
print(tekrar_edenler)



Toplam satır: 3659
         id ProjeNoMan ProjeSiparisNo                       TrafoNo      Tarih  KatBasinaSarim_AG  KanalSayisi_AG  OrtalamaCevre_AG  KatSayisi_AG  IzoleliAxialTelYuksekligi_AG  \
3342   2712  ?DF 18914        2000kVA                           [0] 2015-04-22                1.0               3       1004.023098          12.0                           0.0   
3343   2668  ?DF 18627        2000kVA                           [0] 2015-04-10                1.0               1       1081.526158          13.0                           0.0   
3344   2667  ?DF 18627        2000kVA                           [0] 2015-04-14                1.0               1       1069.960024          14.0                           0.0   
3346   2641  ?DF 18627        2000kVA                           [0] 2015-04-03                1.0               2        955.484795          15.0                           0.0   
3347   2629  ?DF 18627        2000kVA                           [0] 2015-04-03        